In [ ]:
from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool
from google.genai import types

# 1.核心能力简单的功能工具。
# 这遵循将行动与推理分开的最佳实践。
def generate_image(prompt: str) -> dict:
    """
    Generates an image based on a textual prompt.

    Args:
        prompt: A detailed description of the image to generate.

    Returns:
        A dictionary with the status and the generated image bytes.
    """
    print(f"TOOL: Generating image for prompt: '{prompt}'")
    # 在实际实现中，这将调用图像生成 API。
    # 对于此示例，我们返回模拟图像数据。
    mock_image_bytes = b"mock_image_data_for_a_cat_wearing_a_hat"
    return {
        "status": "success",
        # 该工具返回原始字节，代理将处理零件创建。
        "image_bytes": mock_image_bytes,
        "mime_type": "image/png"
    }


# 2. 将 ImageGeneratorAgent 重构为 LlmAgent。
# 现在它可以正确使用传递给它的输入。
image_generator_agent = LlmAgent(
    name="ImageGen",
    model="gemini-2.0-flash",
    description="Generates an image based on a detailed text prompt.",
    instruction=(
        "You are an image generation specialist. Your task is to take the user's request "
        "and use the `generate_image` tool to create the image. "
        "The user's entire request should be used as the 'prompt' argument for the tool. "
        "After the tool returns the image bytes, you MUST output the image."
    ),
    tools=[generate_image]
)

# 3. 将更正后的代理包装在 AgentTool 中。
# 这里的描述是父代理看到的。
image_tool = agent_tool.AgentTool(
    agent=image_generator_agent,
    description="Use this tool to generate an image. The input should be a descriptive prompt of the desired image."
)

# 4. 父代理保持不变。它的逻辑是正确的。
artist_agent = LlmAgent(
    name="Artist",
    model="gemini-2.0-flash",
    instruction=(
        "You are a creative artist. First, invent a creative and descriptive prompt for an image. "
        "Then, use the `ImageGen` tool to generate the image using your prompt."
    ),
    tools=[image_tool]
)

# --- 现在如何运作 ---
# 1. `artist_agent` 决定一个提示，例如，“一只戴着小高帽的逼真猫”。
# 2. 它调用该工具：`ImageGen(input="一只戴着小礼帽的逼真猫。")`
# （注意：AgentTool 使用“input”作为子代理查询的默认参数名称）。
# 3. `agent_tool` 调用 `image_generator_agent`，并以提示作为输入。
# 4. `image_generator_agent` 按照其指示调用 `generate_image(prompt="...")`。
# 5. 该函数返回图像字节。
# 6. `image_generator_agent` 返回最终图像作为其结果。
# 7. `artist_agent` 接收工具调用的图像结果。